In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json

KB_PATH = "/content/drive/MyDrive/setu-guide-rag/data/kb/kb_chunks.json"

with open(KB_PATH) as f:
    kb = json.load(f)

print(f"✅ Loaded {len(kb)} chunks from Day 1")
print("Sample:", kb[0]["chunk_id"], "-", kb[0]["text"][:60], "...")

In [ ]:
!pip install -q sentence-transformers faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
print("✅ Libraries imported")
print("faiss version:", faiss.__version__)

In [ ]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import os

# ---------- Load KB ----------
KB_PATH = "/content/drive/MyDrive/setu-guide-rag/data/kb/kb_chunks.json"
INDEX_PATH = "/content/drive/MyDrive/setu-guide-rag/data/kb/faiss.index"
IDS_PATH = "/content/drive/MyDrive/setu-guide-rag/data/kb/chunk_ids.json"

with open(KB_PATH) as f:
    kb = json.load(f)

print(f"Loaded {len(kb)} chunks")

# ---------- Load model ----------
model = SentenceTransformer("all-MiniLM-L6-v2")

# ---------- Build embedding text ----------
# Include stage and topic so the model has more context
texts = [
    f"[{c['stage']}] {c['topic']}: {c['text']}"
    for c in kb
]

# ---------- Embed ----------
print("Embedding chunks... (this takes ~30 sec)")
embeddings = model.encode(
    texts,
    normalize_embeddings=True,   # so cosine similarity = dot product
    show_progress_bar=True
).astype("float32")

print("Embeddings shape:", embeddings.shape)

# ---------- Build FAISS index ----------
index = faiss.IndexFlatIP(embeddings.shape[1])  # IP = inner product
index.add(embeddings)

print("FAISS index size:", index.ntotal)

# ---------- Save to Drive ----------
faiss.write_index(index, INDEX_PATH)
with open(IDS_PATH, "w") as f:
    json.dump([c["chunk_id"] for c in kb], f, indent=2)

print(f"\n✅ Saved index to {INDEX_PATH}")
print(f"✅ Saved chunk IDs to {IDS_PATH}")

In [ ]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# ---------- Load once at module level ----------
_model = SentenceTransformer("all-MiniLM-L6-v2")
_index = faiss.read_index("/content/drive/MyDrive/setu-guide-rag/data/kb/faiss.index")

with open("/content/drive/MyDrive/setu-guide-rag/data/kb/kb_chunks.json") as f:
    _chunks = json.load(f)

_id_to_chunk = {c["chunk_id"]: c for c in _chunks}


def retrieve(query: str, stage: str, intent: str = None, k: int = 5):
    """
    Stage-aware retrieval over the guidance KB.

    Args:
        query:  the user's question or quick-action text
        stage:  one of the 8 journey stages, or 'global'
        intent: one of conversation_ideas, help_me_understand,
                next_steps, privacy_safety (optional)
        k:      how many chunks to return

    Returns:
        list of dicts: {chunk_id, text, stage, topic, intent, source, score}
    """
    # 1. Embed the query
    q_vec = _model.encode([query], normalize_embeddings=True).astype("float32")

    # 2. Over-fetch then filter by stage and intent
    fetch_n = min(len(_chunks), k * 6)
    scores, idxs = _index.search(q_vec, fetch_n)

    results = []
    for score, i in zip(scores[0], idxs[0]):
        if i == -1:
            continue
        c = _chunks[i]

        # Stage-aware: keep same-stage or global chunks
        if c["stage"] not in (stage, "global"):
            continue

        # Intent filter: keep matching or 'any'
        if intent and c["intent"] not in (intent, "any"):
            continue

        results.append({**c, "score": float(score)})
        if len(results) >= k:
            break

    return results


# ---------- Quick smoke test ----------
print("=== Test 1: conversation ideas at connect stage ===")
for r in retrieve("what should I ask on a first call", "connect", "conversation_ideas", k=3):
    print(f"  [{r['score']:.3f}] {r['chunk_id']} — {r['text'][:70]}")

print("\n=== Test 2: privacy question at understand stage ===")
for r in retrieve("is it safe to share my salary", "understand", "privacy_safety", k=3):
    print(f"  [{r['score']:.3f}] {r['chunk_id']} — {r['text'][:70]}")

print("\n=== Test 3: next steps at discuss stage ===")
for r in retrieve("how do we move forward", "discuss", "next_steps", k=3):
    print(f"  [{r['score']:.3f}] {r['chunk_id']} — {r['text'][:70]}")

In [ ]:
RETRIEVER_SRC = '''"""
retriever.py — Stage-aware RAG retrieval for Setu Guide.
Owner: Pooja Hiwale (AI/ML Intern 2)
"""
import json
import faiss
from sentence_transformers import SentenceTransformer

_model = SentenceTransformer("all-MiniLM-L6-v2")
_index = faiss.read_index("/content/drive/MyDrive/setu-guide-rag/data/kb/faiss.index")

with open("/content/drive/MyDrive/setu-guide-rag/data/kb/kb_chunks.json") as f:
    _chunks = json.load(f)


def retrieve(query: str, stage: str, intent: str = None, k: int = 5):
    """Stage-aware retrieval over the guidance KB."""
    q_vec = _model.encode([query], normalize_embeddings=True).astype("float32")
    fetch_n = min(len(_chunks), k * 6)
    scores, idxs = _index.search(q_vec, fetch_n)

    results = []
    for score, i in zip(scores[0], idxs[0]):
        if i == -1:
            continue
        c = _chunks[i]
        if c["stage"] not in (stage, "global"):
            continue
        if intent and c["intent"] not in (intent, "any"):
            continue
        results.append({**c, "score": float(score)})
        if len(results) >= k:
            break
    return results
'''

SRC_PATH = "/content/drive/MyDrive/setu-guide-rag/src/retriever.py"
with open(SRC_PATH, "w") as f:
    f.write(RETRIEVER_SRC)

print("✅ Saved retrieve function to:", SRC_PATH)

In [ ]:
# Retrieval test cases — 15 scenarios covering all stages and intents
TEST_CASES = [
    # ---- connect ----
    {"q": "what should I ask on a first call",
     "stage": "connect", "intent": "conversation_ideas",
     "expect_topic": "conversation_starters"},
    {"q": "am I ready to move forward",
     "stage": "connect", "intent": "help_me_understand",
     "expect_topic": "checklist"},

    # ---- understand ----
    {"q": "what does the shared readiness meter mean",
     "stage": "understand", "intent": "help_me_understand",
     "expect_topic": "stage_description"},
    {"q": "is it safe to share my salary",
     "stage": "understand", "intent": "privacy_safety",
     "expect_topic": "privacy_consent"},

    # ---- reflect ----
    {"q": "what should I think about before next conversation",
     "stage": "reflect", "intent": "next_steps",
     "expect_topic": "next_steps"},

    # ---- discuss ----
    {"q": "what topics should we discuss",
     "stage": "discuss", "intent": "help_me_understand",
     "expect_topic": "checklist"},
    {"q": "how do we move forward",
     "stage": "discuss", "intent": "next_steps",
     "expect_topic": "next_steps"},

    # ---- agree ----
    {"q": "how do we confirm what we agreed",
     "stage": "agree", "intent": "help_me_understand",
     "expect_topic": "stage_description"},

    # ---- family ----
    {"q": "how should our families meet",
     "stage": "family", "intent": "conversation_ideas",
     "expect_topic": "conversation_starters"},
    {"q": "what should family not see",
     "stage": "family", "intent": "privacy_safety",
     "expect_topic": "privacy_consent"},

    # ---- reflect_again ----
    {"q": "has anything changed after family talks",
     "stage": "reflect_again", "intent": "help_me_understand",
     "expect_topic": "stage_description"},

    # ---- decision ----
    {"q": "should I say yes",
     "stage": "decision", "intent": "help_me_understand",
     "expect_topic": "stage_description"},
    {"q": "how do I share my decision kindly",
     "stage": "decision", "intent": "next_steps",
     "expect_topic": "next_steps"},

    # ---- global safety ----
    {"q": "can you predict if we will succeed",
     "stage": "decision", "intent": "privacy_safety",
     "expect_topic": "safety"},
    {"q": "should I decide for them",
     "stage": "decision", "intent": "privacy_safety",
     "expect_topic": "safety"},
]
print(f"Loaded {len(TEST_CASES)} test cases")

In [ ]:
import time

def run_tests(cases, k=3):
    results = []
    for i, case in enumerate(cases, 1):
        t0 = time.time()
        hits = retrieve(case["q"], case["stage"], case["intent"], k=k)
        elapsed = (time.time() - t0) * 1000  # ms

        topics = [h["topic"] for h in hits]
        stages = [h["stage"] for h in hits]

        # Scoring
        topic_hit = case["expect_topic"] in topics
        stage_ok = all(s in (case["stage"], "global") for s in stages)

        results.append({
            "n": i,
            "query": case["q"],
            "expected_topic": case["expect_topic"],
            "got_topics": topics,
            "topic_hit": topic_hit,
            "stage_ok": stage_ok,
            "latency_ms": round(elapsed, 1),
        })

    return results


results = run_tests(TEST_CASES, k=3)

# Summary
hits = sum(1 for r in results if r["topic_hit"])
stage_pass = sum(1 for r in results if r["stage_ok"])
avg_lat = sum(r["latency_ms"] for r in results) / len(results)

print(f"\n{'='*70}")
print(f"RETRIEVAL EVALUATION — {len(results)} test cases")
print(f"{'='*70}")
print(f"Topic hit rate (expected topic in top-3): {hits}/{len(results)} = {hits/len(results)*100:.0f}%")
print(f"Stage filter correctness:                {stage_pass}/{len(results)} = {stage_pass/len(results)*100:.0f}%")
print(f"Average latency:                         {avg_lat:.1f} ms")

print(f"\n{'='*70}")
print("PER-CASE RESULTS")
print(f"{'='*70}")
for r in results:
    mark = "✅" if r["topic_hit"] and r["stage_ok"] else "❌"
    print(f"{mark} [{r['n']:2}] {r['query'][:45]:<45} "
          f"want={r['expected_topic']:<22} got={r['got_topics']}")

In [ ]:
def consistency_test(query, stage, intent, runs=5):
    results = []
    for _ in range(runs):
        hits = retrieve(query, stage, intent, k=3)
        results.append(tuple(h["chunk_id"] for h in hits))
    return results

print("=== Consistency Test ===")
test_q = ("what should I ask on a first call", "connect", "conversation_ideas")
runs = consistency_test(*test_q, runs=5)
same = all(r == runs[0] for r in runs)
print(f"Query: {test_q[0]}")
print(f"Runs: {runs}")
print(f"All 5 runs identical: {'✅ YES' if same else '❌ NO'}")

In [ ]:
import json

RESULTS_PATH = "/content/drive/MyDrive/setu-guide-rag/data/kb/retrieval_test_results.json"

summary = {
    "total_cases": len(results),
    "topic_hit_rate": f"{hits}/{len(results)}",
    "topic_hit_pct": round(hits/len(results)*100, 1),
    "stage_filter_pass": f"{stage_pass}/{len(results)}",
    "stage_filter_pct": round(stage_pass/len(results)*100, 1),
    "avg_latency_ms": round(avg_lat, 1),
    "results": results,
}

with open(RESULTS_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print(f"✅ Saved results to {RESULTS_PATH}")